# PiGuard: reproducible DGA training and Pi handoff

This notebook uses the same Python modules as the Pi runtime. It trains three lightweight candidates in Python 3.13.5, selects thresholds on validation data, and exports verified bundles. No GPU is required.

Upload `PiGuard_ML_Source.zip` generated by `scripts/package_source.py`. Household DNS records must remain local. Candidate validation results are not final test results. The final evaluation command is gated on stable-power Pi benchmark evidence.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith('.zip')), None)
if not archive:
    raise ValueError('Upload PiGuard_ML_Source.zip')
root = Path('/content/PiGuard_ML')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as source:
    for item in source.infolist():
        target = (root / item.filename).resolve()
        if not target.is_relative_to(root.resolve()):
            raise ValueError('Unsafe archive member')
    source.extractall(root)
os.chdir(root)
print('Source ready:', root)


## Match the Pi runtime

The isolated runtime avoids depending on Colab's default Python or preinstalled libraries. The exact package lock is shipped with the source.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv==0.9.5'], check=True)
subprocess.run(['uv', 'python', 'install', '3.13.5'], check=True)
subprocess.run(['uv', 'venv', '.venv', '--python', '3.13.5'], check=True)
python = str(root / '.venv/bin/python')
subprocess.run(['uv', 'pip', 'install', '--python', python, '-r', 'requirements.lock'], check=True)
subprocess.run([python, '-m', 'pytest', 'test_features.py', 'tests', '-q'], check=True)


## Download the pinned public dataset

The download is checked against the inspected snapshot. Cleaning groups by the effective label before the 70/15/15 split. Conflicting groups are removed. The source lacks DGA-family labels, so this does not test unseen-family detection.


In [ ]:
import urllib.request
url = 'https://raw.githubusercontent.com/tisage/DGA-dataset/b9c75bc92d16736875f094b05180befb163d756a/domains_features.gz'
expected_sha256 = 'acba69ed939da1f13604dab21f15a59e162c379af374fe44176df3f5be468b7a'
dataset = root / 'data/domains_features.gz'
urllib.request.urlretrieve(url, dataset)
if hashlib.sha256(dataset.read_bytes()).hexdigest() != expected_sha256:
    raise ValueError('Dataset hash mismatch')
print('Verified public dataset')


## Train and compare candidates

Use a new output directory for every run. The threshold maximizes validation recall under a 1% measured validation false-positive limit. No-alert candidates are rejected. The test partition stays untouched.


In [ ]:
run = root / 'runs/colab-run-001'
subprocess.run([python, '-m', 'piguard.training', 'train', '--dataset', str(dataset), '--output', str(run)], check=True)
summary = json.loads((run / 'candidates.json').read_text())
import pandas as pd
pd.DataFrame({name: record.get('validation', {}) for name, record in summary.items()}).T


## Download the run and benchmark on the Pi

The archive includes candidate bundles, split files, data audit, and training recipe. Transfer it privately to the Pi and run:

```bash
python -m piguard.benchmark --run runs/colab-run-001 --output benchmark-pi.json
```

Install each candidate's `requirements-pi.txt` in the matching Python environment. Keep the exported manifest digest through the trusted transfer channel. Active undervoltage or throttling invalidates performance acceptance, even if predictions match.


In [ ]:
import shutil
archive_path = shutil.make_archive('/content/piguard-training-run', 'zip', root_dir=root, base_dir='runs/colab-run-001')
files.download(archive_path)


## Final evaluation after Pi acceptance

Run this separately after stable-power benchmark evidence exists. Select the candidate by the declared validation recall / precision / memory ordering among candidates meeting the resource limits. The command enforces this policy and consumes the test set once:

```bash
python -m piguard.training evaluate \
  --run runs/colab-run-001 \
  --candidate SELECTED_CANDIDATE \
  --benchmark benchmark-pi.json \
  --output releases/final-001
```

The release contains test metrics, confidence intervals, ROC/PR/confusion plots, error examples, feature contract, and reference vectors. Final local alert burden still requires a separate seven-day observation period. Do not automatically block domains or treat a score as proof of infection.
